# Atom Training on Google Colab

This notebook runs the full Atom training workflow in Colab with a persistent Drive-backed repo cache.

It is designed to work with `colab_bootstrap.sh` in this repository.


## 1) Mount Google Drive

Drive is used for both persistent source cache and training outputs/checkpoints.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2) Configure repo and paths

Set your GitHub URL and preferred branch once. Keep defaults for the rest unless you need custom paths.


In [ ]:
import os

# Required on first run
os.environ['ATOM_REPO_URL'] = 'https://github.com/<org>/<repo>.git'

# Optional
os.environ['ATOM_BRANCH'] = 'main'
os.environ['ATOM_DRIVE_REPO'] = '/content/drive/MyDrive/dev/atom'
os.environ['ATOM_WORK_REPO'] = '/content/atom'
os.environ['ATOM_INSTALL_JAX_CUDA'] = '1'  # 1 = install CUDA JAX wheel

for key in ['ATOM_REPO_URL', 'ATOM_BRANCH', 'ATOM_DRIVE_REPO', 'ATOM_WORK_REPO', 'ATOM_INSTALL_JAX_CUDA']:
    print(f"{key}={os.environ[key]}")


## 3) Clone (if needed) and bootstrap session

This cell:
- clones repo to `/content/atom` if missing,
- updates Drive cache with `git pull`,
- syncs Drive cache to local runtime,
- installs dependencies.


In [ ]:
%%bash
set -euo pipefail

WORK_REPO="${ATOM_WORK_REPO:-/content/atom}"
REPO_URL="${ATOM_REPO_URL:-}"

if [[ ! -d "$WORK_REPO/.git" ]]; then
  if [[ -z "$REPO_URL" ]]; then
    echo "ERROR: ATOM_REPO_URL must be set before first clone."
    exit 1
  fi
  git clone "$REPO_URL" "$WORK_REPO"
fi

cd "$WORK_REPO"
chmod +x colab_bootstrap.sh
bash colab_bootstrap.sh


## 4) Sanity checks


In [ ]:
%%bash
set -euo pipefail
cd "${ATOM_WORK_REPO:-/content/atom}"

python - <<'PY'
import torch
from src.training.utils.runtime_platform import detect_runtime_platform
print('torch.cuda.is_available =', torch.cuda.is_available())
print('detected runtime platform =', detect_runtime_platform())
PY

nvidia-smi || true


## 5) Quick smoke test (recommended)


In [ ]:
%%bash
set -euo pipefail
cd "${ATOM_WORK_REPO:-/content/atom}"

python train_progressive.py \
  --mode quick \
  --device cuda \
  --use-vmap \
  --output-dir /content/drive/MyDrive/atom_runs/quick_test


## 6) Full training run

Adjust timesteps/population/generations as needed.


In [ ]:
%%bash
set -euo pipefail
cd "${ATOM_WORK_REPO:-/content/atom}"

python train_progressive.py \
  --mode complete \
  --device cuda \
  --use-vmap \
  --timesteps 2000000 \
  --population 8 \
  --generations 10 \
  --output-dir /content/drive/MyDrive/atom_runs/run1


## 7) Resume after disconnect/interruption


In [ ]:
%%bash
set -euo pipefail
cd "${ATOM_WORK_REPO:-/content/atom}"

python resume_population_training.py \
  --checkpoint-dir /content/drive/MyDrive/atom_runs/run1 \
  --start-gen 8 \
  --total-gens 20 \
  --use-vmap


## Troubleshooting

- If GPU is unavailable in this runtime, go to Runtime -> Change runtime type -> GPU.
- If first clone fails with auth, use a public URL or configure a token-auth URL.
- Always write outputs to Drive paths (`/content/drive/MyDrive/...`) to persist results.
